In [15]:
import os 

In [16]:
%pwd

'/Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization'

In [17]:
os.chdir("..")

In [18]:
%pwd

'/Users/akashkumarsinha/Desktop/Text_summerization'

In [19]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt_dir: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    save_steps: float
    gradient_accumulation_steps: int
    report_to: str 


In [20]:
from TextSummarizer.utils.common import read_yaml, create_directories
from TextSummarizer.constants import *


In [21]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        model_trainer_config = self.config.model_trainer
        training_params = self.params.TrainingArguments

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(model_trainer_config.root_dir),
            data_path=Path(model_trainer_config.data_path),
            model_ckpt_dir=Path(model_trainer_config.model_ckpt_dir),
            num_train_epochs=training_params.num_train_epochs,
            warmup_steps=training_params.warmup_steps,
            per_device_train_batch_size=training_params.per_device_train_batch_size,
            per_device_eval_batch_size=training_params.per_device_eval_batch_size,
            weight_decay=training_params.weight_decay,
            logging_steps=training_params.logging_steps,
            save_steps=training_params.save_steps,
            gradient_accumulation_steps=training_params.gradient_accumulation_steps,
            report_to=training_params.report_to,
            
        )

        return model_trainer_config

In [22]:
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from TextSummarizer.custom_logging import logger
from datasets import load_dataset, load_from_disk
import torch

/Users/akashkumarsinha/anaconda3/envs/textsummerization/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-16 12:06:59,052: INFO: PyTorch version 2.4.1 available.]
[2026-06-16 12:06:59,054: INFO: Disabling Tensorflow because USE_TORCH is set]


In [23]:

class ModelTrainer:
    def __init__(self, config, params):
        self.config = config
        self.params = params

        if torch.backends.mps.is_available():
            self.device = "mps"
        elif torch.cuda.is_available():
            self.device = "cuda"
        else:
            self.device = "cpu"

    def train(self):
        logger.info(f"Using device: {self.device}")

        # ----------------------------
        # Load tokenizer & model
        # ----------------------------
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt_dir,use_fast=False)

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt_dir
        )

        # Memory safety (important for Mac)
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
        model.to(self.device)

        # ----------------------------
        # Load RAW dataset
        # ----------------------------
        dataset = load_from_disk(self.config.data_path)

        # ----------------------------
        # TOKENIZATION (THIS WAS MISSING)
        # ----------------------------
        def preprocess_function(batch):
            inputs = tokenizer(
                batch["dialogue"],
                truncation=True,
                padding="max_length",
                max_length=256,
            )

            with tokenizer.as_target_tokenizer():
                labels = tokenizer(
                    batch["summary"],
                    truncation=True,
                    padding="max_length",
                    max_length=64,
                )

            inputs["labels"] = labels["input_ids"]
            return inputs

        dataset = dataset.map(
            preprocess_function,
            batched=True,
            remove_columns=["id", "dialogue", "summary"],
        )

        # ----------------------------
        # Data collator
        # ----------------------------
        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model
        )

        # ----------------------------
        # TrainingArguments (CORRECT NESTING)
        # ----------------------------
        ta = self.params.TrainingArguments

        training_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=int(ta.num_train_epochs),
            warmup_steps=int(ta.warmup_steps),
            per_device_train_batch_size=int(ta.per_device_train_batch_size),
            per_device_eval_batch_size=int(ta.per_device_eval_batch_size),
            gradient_accumulation_steps=int(ta.gradient_accumulation_steps),
            weight_decay=float(ta.weight_decay),
            logging_steps=int(ta.logging_steps),
            save_steps=int(ta.save_steps),
            report_to="none",
            evaluation_strategy="steps",
            save_total_limit=2,
            remove_unused_columns=False,
        )          

        # ----------------------------
        # Trainer
        # ----------------------------
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
            data_collator=data_collator,
            processing_class=tokenizer,  # replaces deprecated tokenizer=
        )

        # ----------------------------
        # Train
        # ----------------------------
        trainer.train()

        # ----------------------------
        # Save model
        # ----------------------------
        model.save_pretrained(os.path.join(self.config.root_dir, "model_pegasus"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))
        logger.info("Model and tokenizer saved successfully.")

In [24]:
try:
    config_manager = ConfigurationManager()

    model_trainer_config = config_manager.get_model_trainer_config()

    model_trainer = ModelTrainer(
    config=model_trainer_config,
    params=config_manager.params
    )

    model_trainer.train()
except Exception as e:
    logger.exception(e)

[2026-06-16 12:06:59,354: INFO: >>> Trying to open: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/config/config.yaml]
[2026-06-16 12:06:59,365: INFO: YAML file: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/config/config.yaml loaded successfully]
[2026-06-16 12:06:59,366: INFO: >>> Trying to open: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/params.yaml]
[2026-06-16 12:06:59,369: INFO: YAML file: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/params.yaml loaded successfully]
[2026-06-16 12:06:59,369: INFO: Directory created at: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/artifacts]
[2026-06-16 12:06:59,401: INFO: Using device: mps]


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/akashkumarsinha/anaconda3/envs/textsummerization/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
0it [00:00, ?it/s]
/Users/akashkumarsinha/anaconda3/envs/textsummerization/lib/python3.8/site-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the mode

{'train_runtime': 0.0064, 'train_samples_per_second': 0.0, 'train_steps_per_second': 0.0, 'train_loss': 0.0, 'epoch': 0}
[2026-06-16 12:07:18,507: INFO: Model and tokenizer saved successfully.]


In [25]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/pegasus-cnn_dailymail"
)

print(next(model.parameters()).device)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cpu
